In [7]:
import numpy as np
import pandas as pd
import ast

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

demo_d1 = pd.read_csv("DEMOBundesliga1_Cleaned.csv")
demo_d2 = pd.read_csv("DEMOBundesliga2_Cleaned.csv")

for df in [demo_d1, demo_d2]:
    df["Date"] = pd.to_datetime(df["Date"])

    df["H2H_Index"] = (
        df["H2H_HomeWins"] - df["H2H_AwayWins"]
    )

combined_history = pd.concat(
    [demo_d1, demo_d2],
    ignore_index=True
).sort_values("Date")

print("Bundesliga 1:", demo_d1.shape)
print("Bundesliga 2:", demo_d2.shape)

Bundesliga 1: (2133, 68)
Bundesliga 2: (2151, 68)


In [8]:
results_d1 = pd.read_csv("Bundesliga_1_beam_results.csv")
results_d2 = pd.read_csv("Bundesliga_2_beam_results.csv")

best_d1 = results_d1.loc[results_d1["Accuracy"].idxmax()]
best_d2 = results_d2.loc[results_d2["Accuracy"].idxmax()]

beam_features_d1 = list(ast.literal_eval(best_d1["Features"]))
beam_features_d2 = list(ast.literal_eval(best_d2["Features"]))

print("Bundesliga 1 best features:")
print(beam_features_d1)
print("CV Accuracy:", best_d1["Accuracy"])

print("\nBundesliga 2 best features:")
print(beam_features_d2)
print("CV Accuracy:", best_d2["Accuracy"])

Bundesliga 1 best features:
['Away_Last5AvgCornersAgainst', 'Away_Last5AvgRedCardsFor', 'Away_Last5AvgShotsOnTargetAgainst', 'Away_Last5AvgShotsOnTargetFor', 'Away_Last5Points', 'Away_RestDays', 'H2H_Index', 'Home_Last5AvgCornersFor', 'Home_Last5AvgGoalsAgainst', 'Home_Last5AvgGoalsFor', 'Home_Last5AvgRedCardsAgainst', 'Home_Last5Points', 'Home_Prev5SeasonAvgPPG', 'Home_RestDays']
CV Accuracy: 0.542429440028147

Bundesliga 2 best features:
['Away_Last5AvgFoulsAgainst', 'Away_Last5AvgFoulsFor', 'Away_Last5AvgGoalsFor', 'Away_Last5AvgRedCardsFor', 'Away_Last5AvgShotsAgainst', 'Away_Last5AvgShotsOnTargetAgainst', 'Away_Last5AvgShotsOnTargetFor', 'Away_Last5AvgYellowCardsAgainst', 'Away_Last5AvgYellowCardsFor', 'Away_Prev5SeasonAvgPPG', 'Away_PrevSeasonsAvailable', 'Away_RestDays', 'H2H_Draws', 'H2H_HomeWins', 'Home_Last5AvgCornersAgainst', 'Home_Last5AvgCornersFor', 'Home_Last5AvgFoulsAgainst', 'Home_Last5AvgRedCardsAgainst', 'Home_Last5AvgShotsFor', 'Home_Last5AvgShotsOnTargetFor', 'Home

In [9]:
def train_model(df, features):

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            C=1.0
        ))
    ])

    pipeline.fit(df[features], df["FTR"])

    return pipeline


model_d1 = train_model(demo_d1, beam_features_d1)
model_d2 = train_model(demo_d2, beam_features_d2)

print("Both models trained successfully!")

Both models trained successfully!


In [10]:
def calculate_team_features(df, team, date, prefix):

    history = df[
        (
            (df["HomeTeam"] == team) |
            (df["AwayTeam"] == team)
        ) &
        (df["Date"] < date) &
        (df["Season"] == "26-27")
    ].sort_values("Date")

    if len(history) < 5:
        raise ValueError(
            f"Insufficient match history for {team}"
        )

    last5 = history.tail(5)

    is_home = last5["HomeTeam"] == team

    features = {}

    features[f"{prefix}_Last5Points"] = np.where(
        is_home,
        last5["HomePoints"],
        last5["AwayPoints"]
    ).sum()

    stat_mapping = {
        "Goals": ("FTHG", "FTAG"),
        "Shots": ("HS", "AS"),
        "ShotsOnTarget": ("HST", "AST"),
        "Corners": ("HC", "AC"),
        "Fouls": ("HF", "AF"),
        "YellowCards": ("HY", "AY"),
        "RedCards": ("HR", "AR")
    }

    for stat, (home_col, away_col) in stat_mapping.items():

        features[f"{prefix}_Last5Avg{stat}For"] = np.mean(
            np.where(
                is_home,
                last5[home_col],
                last5[away_col]
            )
        )

        features[f"{prefix}_Last5Avg{stat}Against"] = np.mean(
            np.where(
                is_home,
                last5[away_col],
                last5[home_col]
            )
        )

    latest = history.iloc[-1]

    old_prefix = (
        "Home" if latest["HomeTeam"] == team
        else "Away"
    )

    historical_features = [
        "Prev5SeasonAvgPPG",
        "Prev5SeasonAvgPoints",
        "PrevSeasonsAvailable"
    ]

    for feature in historical_features:

        features[f"{prefix}_{feature}"] = (
            latest[f"{old_prefix}_{feature}"]
        )

    features[f"{prefix}_RestDays"] = (
        date - latest["Date"]
    ).days

    return features

In [11]:
def calculate_h2h(home, away, date):

    history = combined_history[
        (
            (
                (combined_history["HomeTeam"] == home) &
                (combined_history["AwayTeam"] == away)
            ) |
            (
                (combined_history["HomeTeam"] == away) &
                (combined_history["AwayTeam"] == home)
            )
        ) &
        (combined_history["Date"] < date)
    ].sort_values("Date").tail(5)

    home_wins = 0
    away_wins = 0
    draws = 0

    for _, match in history.iterrows():

        if match["FTR"] == "D":
            draws += 1
            continue

        winner = (
            match["HomeTeam"]
            if match["FTR"] == "H"
            else match["AwayTeam"]
        )

        if winner == home:
            home_wins += 1
        else:
            away_wins += 1

    return {
        "H2H_HomeWins": home_wins,
        "H2H_AwayWins": away_wins,
        "H2H_Draws": draws,
        "H2H_Matches": len(history),
        "H2H_Index": home_wins - away_wins
    }

In [12]:
def predict_match(home, away, date, division,
                  draw_weight=1.0):

    date = pd.Timestamp(date)

    if division == 1:
        df = demo_d1
        model = model_d1
        features = beam_features_d1

    elif division == 2:
        df = demo_d2
        model = model_d2
        features = beam_features_d2

    else:
        raise ValueError("Division must be 1 or 2")

    if date <= df["Date"].max():
        raise ValueError("Enter an upcoming match date")

    match_features = {}

    match_features.update(
        calculate_team_features(df, home, date, "Home")
    )

    match_features.update(
        calculate_team_features(df, away, date, "Away")
    )

    match_features.update(
        calculate_h2h(home, away, date)
    )

    X = pd.DataFrame([match_features])[features]

    probabilities = model.predict_proba(X)[0]

    classes = model.named_steps["model"].classes_

    # Optional draw adjustment
    probabilities[classes == "D"] *= draw_weight

    probabilities /= probabilities.sum()

    probs = dict(zip(classes, probabilities))

    prediction = max(probs, key=probs.get)

    outcome_names = {
        "H": home,
        "D": "Draw",
        "A": away
    }

    confidence = probs[prediction]

    if confidence >= 0.70:
        level = "High"
    elif confidence >= 0.50:
        level = "Medium"
    else:
        level = "Low"

    return {
        "Date": date.date(),
        "Home Team": home,
        "Away Team": away,
        "Home Win (%)": probs["H"] * 100,
        "Draw (%)": probs["D"] * 100,
        "Away Win (%)": probs["A"] * 100,
        "Prediction": outcome_names[prediction],
        "Confidence (%)": confidence * 100,
        "Confidence Level": level
    }

In [19]:
def interactive_demo():

    print("===== BUNDESLIGA AI MATCH PREDICTOR =====")

    while True:

        division_input = input(
            "\nEnter division (1 or 2), or Q to quit: "
        ).strip()

        if division_input.lower() == "q":
            break

        if division_input not in ["1", "2"]:
            print("Invalid division.")
            continue

        division = int(division_input)

        df = demo_d1 if division == 1 else demo_d2

        current = df[df["Season"] == "26-27"]

        teams = sorted(
            set(current["HomeTeam"]) |
            set(current["AwayTeam"])
        )

        print("\nAvailable teams:")
        print(", ".join(teams))

        # Allow case-insensitive team names
        team_lookup = {team.lower(): team for team in teams}

        home = input("\nEnter home team: ").strip().lower()
        away = input("Enter away team: ").strip().lower()

        if home not in team_lookup or away not in team_lookup:
            print("Invalid team name.")
            continue

        home = team_lookup[home]
        away = team_lookup[away]

        if home == away:
            print("Teams must be different.")
            continue

        match_date = input("Enter match date (YYYY-MM-DD): ")

        try:

            result = predict_match(
                home=home,
                away=away,
                date=match_date,
                division=division,
                draw_weight=1.0
            )

            print("\n" + "=" * 50)
            print(f"BUNDESLIGA {division}")
            print(f"{home} vs. {away}")
            print(f"Match date: {match_date}")
            print("=" * 50)

            print(f"\n{home} Win: {result['Home Win (%)']:.2f}%")
            print(f"Draw: {result['Draw (%)']:.2f}%")
            print(f"{away} Win: {result['Away Win (%)']:.2f}%")

            print(f"\nPredicted outcome: {result['Prediction']}")
            print(f"Confidence: {result['Confidence (%)']:.2f}%")
            print(f"Confidence level: {result['Confidence Level']}")

            print("=" * 50)

        except Exception as e:
            print("Prediction error:", e)


interactive_demo()

===== BUNDESLIGA AI MATCH PREDICTOR =====

Available teams:
Augsburg, Bayern Munich, Dortmund, Ein Frankfurt, Elversberg, FC Koln, Freiburg, Hamburg, Hoffenheim, Leverkusen, M'gladbach, Mainz, Paderborn, RB Leipzig, Schalke 04, Stuttgart, Union Berlin, Werder Bremen

BUNDESLIGA 1
Stuttgart vs. Ein Frankfurt
Match date: 2026-11-28

Stuttgart Win: 45.47%
Draw: 25.94%
Ein Frankfurt Win: 28.58%

Predicted outcome: Stuttgart
Confidence: 45.47%
Confidence level: Low

Available teams:
Augsburg, Bayern Munich, Dortmund, Ein Frankfurt, Elversberg, FC Koln, Freiburg, Hamburg, Hoffenheim, Leverkusen, M'gladbach, Mainz, Paderborn, RB Leipzig, Schalke 04, Stuttgart, Union Berlin, Werder Bremen

BUNDESLIGA 1
Bayern Munich vs. Union Berlin
Match date: 2026-11-28

Bayern Munich Win: 85.64%
Draw: 10.71%
Union Berlin Win: 3.65%

Predicted outcome: Bayern Munich
Confidence: 85.64%
Confidence level: High
Invalid division.
Invalid division.
Invalid division.
Invalid division.
Invalid division.


KeyboardInterrupt: Interrupted by user